# 04 · The FastAPI service, the run log and the evaluation

**What you will learn**
1. Call every endpoint in-process with FastAPI's `TestClient` (same code as `uvicorn pharma_agent.api:app`)
2. Read the ACID run log in SQLite
3. How the evaluation scores the agent, with a small live run and the latest full report

In [1]:
# Setup: make the project importable and quiet the HTTP logs
import sys, logging, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
logging.getLogger("httpx").setLevel(logging.WARNING)

def show(obj, limit=1500):
    """Pretty-print a pydantic model, dict or list, cut to `limit` characters."""
    if hasattr(obj, "model_dump"):
        obj = obj.model_dump()
    if isinstance(obj, list):
        obj = [o.model_dump() if hasattr(o, "model_dump") else o for o in obj]
    text = json.dumps(obj, indent=2, ensure_ascii=False, default=str)
    print(text if len(text) <= limit else text[:limit] + "\n... (cut)")

In [2]:
from fastapi.testclient import TestClient
from pharma_agent.api import create_app
from pharma_agent.config import load_config
from pharma_agent.service import PharmaAgent
from pharma_agent.store import Store

cfg = load_config(overrides={"store.path": ":memory:"})
store = Store(":memory:")

async def factory():
    return await PharmaAgent.create(cfg, store=store)

client = TestClient(create_app(factory))
client.__enter__()          # runs the app's startup (builds the agent once)
print(client.get("/health").json())

{'status': 'ok', 'llm_provider': 'groq', 'tool_mode': 'direct'}


In [3]:
for t in client.get("/tools").json():
    print(f"{t['name']:26s} {list(t['args'])}")
print()
for a in client.get("/agents").json():
    print(f"{a['name']:17s} {a['tools']}")

search_drug_labels         ['drug', 'section', 'limit']
get_adverse_event_counts   ['drug', 'top_n']
search_clinical_trials     ['condition', 'intervention', 'status', 'limit']
search_pubmed              ['query', 'limit']

label_agent       ['search_drug_labels', 'get_adverse_event_counts']
trials_agent      ['search_clinical_trials']
literature_agent  ['search_pubmed']


## `POST /ask`: the structured, cited answer

In [4]:
body = client.post("/ask", json={"question": "What are the contraindications for apixaban?"}).json()
print(body["answer"]["answer"], "\n")
for p in body["answer"]["key_points"]:
    print("-", p["text"], p["citations"])
print("\nGuardrail:", body["guardrail"], "| latency:", body["latency_ms"], "ms")

Apixaban is contraindicated in patients with active pathological bleeding and in those who have had a severe hypersensitivity reaction to the drug, such as anaphylaxis. 

- Active pathological bleeding is a contraindication for apixaban. ['fda-label:41a133ef:contraindications']
- Severe hypersensitivity reactions, including anaphylaxis, to apixaban are also contraindications. ['fda-label:41a133ef:contraindications']

Guardrail: {'evidence_count': 1, 'valid_citations': 2, 'invalid_citations': [], 'repaired_citations': [], 'unsupported_points': 0} | latency: 120429.3 ms


In [5]:
print("Validation:", client.post("/ask", json={"question": "hi"}).status_code, "(422 = rejected, too short)")
print(client.get("/runs").json())

Validation: 422 (422 = rejected, too short)
[{'id': 'c2e60862114c', 'question': 'What are the contraindications for apixaban?', 'latency_ms': 120429.3, 'created_at': '2026-11-25T19:47:52+00:00', 'evidence': 1}]


## The run log is ACID
`save_run` writes the run and all its evidence rows in **one transaction**: both land, or neither does.
The tests (`tests/test_store_acid.py`) force a failure halfway through and check nothing is left behind.

In [6]:
run = store.get_run(body["run_id"])
print(run["question"], "|", len(run["evidence"]), "evidence rows |", run["created_at"])
print("runs:", store.count("runs"), " run_evidence:", store.count("run_evidence"))
client.__exit__(None, None, None)   # stop the TestClient (and its event loop) before the next part

What are the contraindications for apixaban? | 1 evidence rows | 2026-11-25T19:47:52+00:00
runs: 1  run_evidence: 1


## Evaluation
Each question in `eval/questions.jsonl` lists the agents and sources it should need. The harness measures:

| Metric | Meaning |
|---|---|
| routing_exact | supervisor chose exactly the expected agents |
| source_recall | expected sources returned evidence |
| citation_precision | cited ids that were real evidence (before the guardrail) |
| grounded_points | key points with at least one valid citation |
| term_hit | expected key terms appear in the answer |

A small live run (2 questions):

In [7]:
from pharma_agent.evaluation import load_questions, run_eval
questions = load_questions(ROOT / "eval" / "questions.jsonl")
# Build a separate agent here: the TestClient's agent lives on the TestClient's own event loop
# (in another thread), and awaiting it from this notebook's loop would deadlock.
eval_agent = await PharmaAgent.create(cfg, store=store)
mini = await run_eval(eval_agent, [questions[0], questions[12]], delay_seconds=10)
show(mini["summary"])

[1/2] q01: routing=1 cit_prec=1.00 7.5s


[2/2] q13: routing=1 cit_prec=1.00 25.8s
{
  "questions": 2,
  "answered": 2,
  "errors": 0,
  "routing_exact": 1.0,
  "routing_recall": 1.0,
  "source_recall": 1.0,
  "citation_precision": 1.0,
  "grounded_points": 1.0,
  "term_hit": 1.0,
  "latency_p50_s": 16.7,
  "latency_p95_s": 25.8
}


And the latest full run, saved by `python -m pharma_agent eval`:

In [8]:
latest = sorted((ROOT / "eval" / "results").glob("eval_*.json"))[-1]
report = json.loads(latest.read_text(encoding="utf-8"))
print(latest.name)
show(report["summary"])
print()
for r in report["rows"]:
    s = r.get("scores", {})
    print(f"{r['id']}  routing={s.get('routing_exact', '-')}  sources={s.get('source_recall', '-')}  "
          f"cit_prec={s.get('citation_precision', '-')}  terms={s.get('term_hit', '-')}  {r.get('error', '')[:60]}")

eval_20261126_010042_rescored.json
{
  "questions": 16,
  "answered": 16,
  "errors": 0,
  "routing_exact": 1.0,
  "routing_recall": 1.0,
  "source_recall": 1.0,
  "citation_precision": 1.0,
  "grounded_points": 1.0,
  "term_hit": 1.0,
  "latency_p50_s": 5.9,
  "latency_p95_s": 43.3
}

q01  routing=1.0  sources=1.0  cit_prec=1.0  terms=1.0  
q02  routing=1.0  sources=1.0  cit_prec=1.0  terms=1.0  
q03  routing=1.0  sources=1.0  cit_prec=1.0  terms=1.0  
q04  routing=1.0  sources=1.0  cit_prec=1.0  terms=1.0  
q05  routing=1.0  sources=1.0  cit_prec=1.0  terms=1.0  
q06  routing=1.0  sources=1.0  cit_prec=1.0  terms=1.0  
q07  routing=1.0  sources=1.0  cit_prec=1.0  terms=1.0  
q08  routing=1.0  sources=1.0  cit_prec=1.0  terms=1.0  
q09  routing=1.0  sources=1.0  cit_prec=1.0  terms=1.0  
q10  routing=1.0  sources=1.0  cit_prec=1.0  terms=1.0  
q11  routing=1.0  sources=1.0  cit_prec=1.0  terms=1.0  
q12  routing=1.0  sources=1.0  cit_prec=1.0  terms=1.0  
q13  routing=1.0  sources=1.0